# HYPERPARAMETER TUNING AND ENSEBLING METHODS

## Setup and Data Loading

### Imports

In [1]:
#sklearn
import sklearn
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import make_scorer, cohen_kappa_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.base import clone
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Metrics
from sklearn.metrics import f1_score, make_scorer

# Imbalanced-learn (SMOTE, Oversampling)
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.pipeline import make_pipeline

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optuna
import optuna
import optuna.visualization as vis

# Preprocessing
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessorLOEEALL

import numpy as np
import pandas as pd

#Warnings lightGBM
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

import time


c:\Users\asier.gomez\AppData\Local\anaconda3\envs\petfinder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset loading

In [37]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

categorical_cols = ['Type', 'Gender', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'RescuerID', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength','Breed1', 'Breed2', 'State']

(10045, 32)
(10045,)
(4948, 32)
(4948,)


### Initialization

In [3]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializng the second evaluation technique
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
scoring = {
        "f1_macro": "f1_macro",
        "QWK": qwk
    }

## Optuna hyperparameter tuning

### XGBoost hyperparameter tuning

In [ ]:
"""
def objective_xgb(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.05)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)
    
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss"    # for multiple classification
    }

    #Creamos pipeline con preprocessor y SMOTE
    pipe_base = imb_make_pipeline(
        preprocessorLOEEALL,
        RandomOverSampler(random_state=42),      
        XGBClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
xgb_study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study2",
    storage="sqlite:///optuna_studies/xgb_study.db",  # se guarda en este archivo
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

#we enter the parameters with the best result in the last study
#If it is the first study we put the default parameters
xgb_study.enqueue_trial({
    "looe_sigma": 0.028238316352854904,
    "n_estimators": 976,
    "max_depth": 9,
    "learning_rate": 0.014031401130172998,
    "min_child_weight": 6,
    "subsample": 0.7395366759188357,
    "colsample_bytree": 0.8063729967181755
})

xgb_study.optimize(objective_xgb, n_trials=100)

print(f"Mejores parámetros: {xgb_study.best_params}")
"""

[I 2026-03-14 07:49:57,923] A new study created in RDB with name: xgb_study2
[I 2026-03-14 07:51:06,602] Trial 0 finished with value: 0.4176029298921349 and parameters: {'looe_sigma': 0.028238316352854904, 'n_estimators': 976, 'max_depth': 9, 'learning_rate': 0.014031401130172998, 'subsample': 0.7395366759188357, 'colsample_bytree': 0.8063729967181755, 'min_child_weight': 6}. Best is trial 0 with value: 0.4176029298921349.
[I 2026-03-14 07:51:23,413] Trial 1 finished with value: 0.37407660847475077 and parameters: {'looe_sigma': 0.008935538094064511, 'n_estimators': 486, 'max_depth': 5, 'learning_rate': 0.19446566339847182, 'subsample': 0.8282609244084409, 'colsample_bytree': 0.8031070919429738, 'min_child_weight': 1}. Best is trial 0 with value: 0.4176029298921349.
[I 2026-03-14 07:52:25,611] Trial 2 finished with value: 0.40661420869910553 and parameters: {'looe_sigma': 0.010802489415310224, 'n_estimators': 1094, 'max_depth': 10, 'learning_rate': 0.0372574328386415, 'subsample': 0.77

Mejores parámetros: {'looe_sigma': 0.010773764065091765, 'n_estimators': 552, 'max_depth': 10, 'learning_rate': 0.01294031155889707, 'subsample': 0.6459278279430541, 'colsample_bytree': 0.6169935895547909, 'min_child_weight': 1}


In [46]:
xgb_study = optuna.load_study(
    study_name="xgb_study2", 
    storage="sqlite:///optuna_studies/xgb_study.db" 
)
fig = vis.plot_param_importances(xgb_study)
fig.show()

In [21]:
#---Trial with the optimum model---
xgb_params = xgb_study.best_params.copy()
print(xgb_params)
loe_sigma_xgb = xgb_params.pop("looe_sigma", None)
xgb_best = XGBClassifier(
    **xgb_params,
    random_state=42,         
    n_jobs=-1,
    eval_metric="mlogloss"  
)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_xgb)
pipe_xgb = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    xgb_best
)
score_cv = cross_validate(pipe_xgb, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK xgb optimum: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the underfitted model---
xgb_best.set_params(max_depth = 2, learning_rate = 0.01)
pipe_xgb = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    xgb_best
)
score_cv = cross_validate(pipe_xgb, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK xgb underfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the overfitted model---
xgb_best.set_params(max_depth = 15, learning_rate = 0.3)
pipe_xgb = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    xgb_best
)
score_cv = cross_validate(pipe_xgb, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK xgb overfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

{'looe_sigma': 0.010773764065091765, 'n_estimators': 552, 'max_depth': 10, 'learning_rate': 0.01294031155889707, 'subsample': 0.6459278279430541, 'colsample_bytree': 0.6169935895547909, 'min_child_weight': 1}
QWK xgb optimum: train(0.6757564004938812), test(0.4201418518609101)
QWK xgb underfitted: train(0.478465396551713), test(0.35653233654908967)
QWK xgb overfitted: train(0.6621236830109449), test(0.3839637094395907)


### XGBoost hyperparameter tuning balanced

In [4]:
def objective_xgb_balanced(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.05)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)
    
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss"    # for multiple classification
    }

    pipe_base = imb_make_pipeline(
        preprocessorLOEEALL,     
        XGBClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
xgb_study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study_bal",
    storage="sqlite:///optuna_studies/xgb_study_bal.db",  # se guarda en este archivo
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

#we enter the parameters with the best result in the last study
#If it is the first study we put the default parameters
xgb_study.enqueue_trial({
    "looe_sigma": 0.05,           
    "n_estimators": 100,        
    "max_depth": 6,              
    "learning_rate": 0.3,         
    "min_child_weight": 1,       
    "subsample": 1.0,             
    "colsample_bytree": 1.0       
})

xgb_study.optimize(objective_xgb_balanced, n_trials=100)

print(f"Mejores parámetros: {xgb_study.best_params}")

[I 2026-03-16 17:04:30,722] A new study created in RDB with name: xgb_study_bal
[I 2026-03-16 17:04:36,375] Trial 0 finished with value: 0.37895698140221795 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 1}. Best is trial 0 with value: 0.37895698140221795.
[I 2026-03-16 17:05:12,397] Trial 1 finished with value: 0.37579710925668836 and parameters: {'looe_sigma': 0.021458584772302353, 'n_estimators': 878, 'max_depth': 9, 'learning_rate': 0.1775252598705841, 'subsample': 0.711621905515208, 'colsample_bytree': 0.7018564047078526, 'min_child_weight': 9}. Best is trial 0 with value: 0.37895698140221795.
[I 2026-03-16 17:05:48,364] Trial 2 finished with value: 0.3898942111837062 and parameters: {'looe_sigma': 0.009940901109897938, 'n_estimators': 950, 'max_depth': 6, 'learning_rate': 0.16415764669879912, 'subsample': 0.6799833100941418, 'colsample_bytree': 0.7912945812052463, 'min_

Mejores parámetros: {'looe_sigma': 0.024366032562905083, 'n_estimators': 441, 'max_depth': 4, 'learning_rate': 0.043177464816704605, 'subsample': 0.8087435052433899, 'colsample_bytree': 0.9094862583458461, 'min_child_weight': 10}


### CatBoost hyperparameter tuning

In [ ]:
"""
def objective_catboost(trial):
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "random_strength": 1,
        "bagging_temperature": 1,
        "border_count": 128,
        "verbose":0, 
        "cat_features": categorical_cols,
        "allow_writing_files":False,
        "random_state": 42,
        "thread_count": -1  
    }

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(X_fold_train, y_fold_train)
        preds = model.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
cat_study = optuna.create_study(
    direction="maximize",
    study_name="cat_study_native",
    storage="sqlite:///optuna_studies/cat_study.db",  # se guarda en este archivo
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

cat_study.enqueue_trial({
    "n_estimators": 1000,         
    "max_depth": 6,               
    "learning_rate": 0.03,        
    "l2_leaf_reg": 3.0,
})

cat_study.optimize(objective_catboost, n_trials=100)

print(f"Mejores parámetros: {cat_study.best_params}")
"""

[I 2026-03-13 23:46:37,081] Using an existing study with name 'cat_study_native' instead of creating a new one.
[I 2026-03-13 23:54:37,610] Trial 1 finished with value: 0.4433665335151252 and parameters: {'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-13 23:58:46,230] Trial 2 finished with value: 0.43588281142792384 and parameters: {'n_estimators': 657, 'max_depth': 5, 'learning_rate': 0.12889699430870627, 'l2_leaf_reg': 2.971221171040712}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-14 00:02:04,109] Trial 3 finished with value: 0.436811080806741 and parameters: {'n_estimators': 421, 'max_depth': 6, 'learning_rate': 0.11785770555979118, 'l2_leaf_reg': 6.451872448595916}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-14 00:05:38,682] Trial 4 finished with value: 0.43909749153721833 and parameters: {'n_estimators': 465, 'max_depth': 6, 'learning_rate': 0.0947608269863

Mejores parámetros: {'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}


In [5]:
cat_study = optuna.load_study(
    study_name="cat_study_native", 
    storage="sqlite:///optuna_studies/cat_study.db" 
)
fig = vis.plot_param_importances(cat_study)
fig.show()

In [44]:
#---Trial with the optimum model---
cat_params = cat_study.best_params.copy()

cat_best = CatBoostClassifier(
    **cat_params,
    random_strength=1,
    bagging_temperature=1,
    border_count=128,
    allow_writing_files=False,
    random_state=42,
    thread_count=-1,
    verbose=0,
)

pipe_cat = make_pipeline(
    cat_best
)

score_cv = cross_validate(cat_best, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True, params={'cat_features': categorical_cols})

print(f"QWK cat optimum: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the underfitted model---
cat_best.set_params(n_estimators = 100, learning_rate = 0.01)

score_cv = cross_validate(cat_best, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True, params={'cat_features': categorical_cols})
print(f"QWK cat underfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the overfitted model---
cat_best.set_params(n_estimators = 1000, learning_rate = 0.3)

score_cv = cross_validate(cat_best, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True, params={'cat_features': categorical_cols})
print(f"QWK cat overfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

QWK cat optimum: train(0.752375664144535), test(0.4433665335151252)
QWK cat underfitted: train(0.6218834931070287), test(0.3580352275091859)
QWK cat overfitted: train(0.8417120611645649), test(0.42674089403638976)


### RandomForest hyperparameter tuning

In [ ]:
"""
def objective_rf(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.06)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_int("max_depth", 10, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "bootstrap": True,
        "class_weight": "balanced", 
        "random_state": 42,
        "n_jobs": -1
    }

    pipe_base = make_pipeline(
        preprocessorLOEEALL,
        RandomForestClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# Crear estudio de Optuna
rf_study = optuna.create_study(
    direction="maximize",
    study_name="rf_study2",
    storage="sqlite:///optuna_studies/rf_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

#We use the best hyperparameters of the previous study
#If it is the first study we use the default parameters
rf_study.enqueue_trial({
    "looe_sigma": 0.013264638423263085,
    "n_estimators": 505,
    "max_depth": 60,
    "min_samples_split": 11,
    "min_samples_leaf": 1,
    "max_features": "sqrt"
})

# Lanzar optimización
rf_study.optimize(objective_rf, n_trials=100)

# Resultados
print(f"Mejores parámetros: {rf_study.best_params}")
"""

[I 2026-03-13 18:30:45,042] A new study created in RDB with name: rf_study2
[I 2026-03-13 18:30:56,111] Trial 0 finished with value: 0.41728014610265063 and parameters: {'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:07,591] Trial 1 finished with value: 0.40168592536987396 and parameters: {'looe_sigma': 0.03883443450450948, 'n_estimators': 508, 'max_depth': 21, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:11,808] Trial 2 finished with value: 0.39955385872732274 and parameters: {'looe_sigma': 0.037987927251354234, 'n_estimators': 144, 'max_depth': 71, 'min_samples_split': 20, 'min_samples_leaf': 13, 'max_features': 'log2'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:18,581] Trial 3 finished with valu

Mejores parámetros: {'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [17]:
rf_study = optuna.load_study(
    study_name="rf_study2", 
    storage="sqlite:///optuna_studies/rf_study.db" 
)
fig = vis.plot_param_importances(rf_study)
fig.show()

In [ ]:
#---Trial with the optimum model---
rf_params = rf_study.best_params.copy()
print(rf_params)
loe_sigma_rf = rf_params.pop("looe_sigma", None)
rf_best = RandomForestClassifier(
    **rf_params,    
    bootstrap=True,            
    class_weight="balanced",    
    random_state=42,
    n_jobs=-1
)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_rf)
pipe_rf = make_pipeline(
    clone(preprocessorLOEEALL),
    rf_best
)
score_cv = cross_validate(pipe_rf, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK rf optimum: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the underfitted model---
rf_best.set_params(n_estimators = 80, min_samples_split = 50)
pipe_rf = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    rf_best
)
score_cv = cross_validate(pipe_rf, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK rf underfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the overfitted model---
rf_best.set_params(n_estimators = 800, min_samples_split = 2)
pipe_rf = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    rf_best
)
score_cv = cross_validate(pipe_rf, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK rf overfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

{'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
QWK rf optimum: train(0.6586526363597247), test(0.41728014610265063)
QWK rf underfitted: train(0.5679180966596385), test(0.39489387054211)
QWK rf overfitted: train(0.7458817786979864), test(0.4118110873922654)


### LightGBM hyperparameter tuning

In [ ]:
"""
def objective_lgbm_balanced(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.1)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7 , 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }

    # num_leaves <= 2^(max_depth) - 1
    # Forzamos a que si Optuna elige una mala combinación, se corrija sola:
    max_leaves_allowed = (2 ** params["max_depth"]) - 1
    params["num_leaves"] = min(params["num_leaves"], max_leaves_allowed)

    # Creamos pipeline clásico de Scikit-Learn (sin SMOTE)
    pipe_base = imb_make_pipeline(
        preprocessorLOEEALL,
        RandomOverSampler(random_state = 42),
        LGBMClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)


# Crear estudio de Optuna (con nombre nuevo para no pisar el anterior)
lgbm_study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_study1",
    storage="sqlite:///optuna_studies/lgbm_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

# Trial 0 inicial razonable
lgbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,          
    "max_depth": 15,              
    "num_leaves": 31,             
    "learning_rate": 0.1,        
    "min_child_samples": 20,      
    "subsample": 1.0,             
    "colsample_bytree": 1.0       
})

# Lanzar optimización
lgbm_study.optimize(objective_lgbm_balanced, n_trials=100)

# Resultados
print(f"Mejores parámetros: {lgbm_study.best_params}")
"""

[I 2026-03-13 18:46:46,523] A new study created in RDB with name: lgbm_study1
[I 2026-03-13 18:46:51,427] Trial 0 finished with value: 0.39194816689722767 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 15, 'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 1.0}. Best is trial 0 with value: 0.39194816689722767.
[I 2026-03-13 18:47:02,920] Trial 1 finished with value: 0.3799719119654584 and parameters: {'looe_sigma': 0.00784130618090908, 'n_estimators': 161, 'max_depth': 12, 'learning_rate': 0.2645936406421905, 'num_leaves': 56, 'min_child_samples': 10, 'subsample': 0.9938652830131463, 'colsample_bytree': 0.6241313538255718}. Best is trial 0 with value: 0.39194816689722767.
[I 2026-03-13 18:47:43,562] Trial 2 finished with value: 0.37859275945447596 and parameters: {'looe_sigma': 0.010098478722223747, 'n_estimators': 650, 'max_depth': 11, 'learning_rate': 0.17283581985646962, 'num_leaves': 91, 'min_child_samples'

Mejores parámetros: {'looe_sigma': 0.017274353532403468, 'n_estimators': 431, 'max_depth': 14, 'learning_rate': 0.021117244921237944, 'num_leaves': 49, 'min_child_samples': 8, 'subsample': 0.7392171099865767, 'colsample_bytree': 0.9999202226086774}


In [45]:
lgbm_study = optuna.load_study(
    study_name="lgbm_study1", 
    storage="sqlite:///optuna_studies/lgbm_study.db" 
)
fig = vis.plot_param_importances(lgbm_study)
fig.show()

In [ ]:
#---Trial with the optimum model---
lgbm_params = lgbm_study.best_params.copy()
print(lgbm_params)
loe_sigma_lgbm = lgbm_params.pop("looe_sigma", None)
lgbm_best = LGBMClassifier(
    **lgbm_params,  
    subsample_freq=1,          
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_lgbm)
pipe_lgbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42), 
    lgbm_best
)
score_cv = cross_validate(pipe_lgbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK lgbm optimum: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the underfitted model---
lgbm_best.set_params(n_estimators = 100, min_samples_split = 50)
pipe_lgbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    lgbm_best
)
score_cv = cross_validate(pipe_lgbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK lgbm underfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the overfitted model---
lgbm_best.set_params(n_estimators = 1000, min_samples_split = 1)
pipe_lgbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    lgbm_best
)
score_cv = cross_validate(pipe_lgbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK lgbm overfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

{'looe_sigma': 0.017274353532403468, 'n_estimators': 431, 'max_depth': 14, 'learning_rate': 0.021117244921237944, 'num_leaves': 49, 'min_child_samples': 8, 'subsample': 0.7392171099865767, 'colsample_bytree': 0.9999202226086774}
QWK lgbm optimum: train(0.6509658637544435), test(0.42338340935311647)
QWK lgbm underfitted: train(0.5779853829246011), test(0.3946060796748788)
QWK lgbm overfitted: train(0.6729993207276334), test(0.40979262264048427)


### Gradient Boosting hyperparameter tuning

In [ ]:
"""
def objective_gbm(trial):
    # 1. Rango ajustado para tu target multiclase (0 a 4)
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)
    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 6), 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0), 
        "random_state": 42
    }

    # 3. Pipeline normal (make_pipeline) SIN técnicas de balanceo
    pipe_base = make_pipeline(
        preprocessorLOEEALL,
        GradientBoostingClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)


# Crear estudio de Optuna
gbm_study = optuna.create_study(
    direction="maximize",
    study_name="gbm_study",
    storage="sqlite:///optuna_studies/gbm_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

# Valores por defecto reales de GradientBoostingClassifier
gbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.1,
    "min_samples_leaf": 1,
    "subsample": 1.0,
    "max_features": 1.0
})

# Lanzar optimización
gbm_study.optimize(objective_gbm, n_trials=100)

# Resultados
print(f"Mejores parámetros: {gbm_study.best_params}")
"""

[I 2026-03-13 20:30:38,864] A new study created in RDB with name: gbm_study
[I 2026-03-13 20:32:15,581] Trial 0 finished with value: 0.3994245456952566 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'min_samples_leaf': 1, 'subsample': 1.0, 'max_features': 1.0}. Best is trial 0 with value: 0.3994245456952566.
[I 2026-03-13 20:33:54,969] Trial 1 finished with value: 0.3300455048466535 and parameters: {'looe_sigma': 0.2012208862374252, 'n_estimators': 102, 'max_depth': 5, 'learning_rate': 0.11339828749227336, 'min_samples_leaf': 10, 'subsample': 0.8116584059118779, 'max_features': 0.7561180571627557}. Best is trial 0 with value: 0.3994245456952566.
[I 2026-03-13 20:38:07,122] Trial 2 finished with value: 0.39042108115938745 and parameters: {'looe_sigma': 0.06135296511275576, 'n_estimators': 314, 'max_depth': 6, 'learning_rate': 0.03964487902594386, 'min_samples_leaf': 44, 'subsample': 0.6233148013999327, 'max_features': 0.7262514297438959}.

Mejores parámetros: {'looe_sigma': 0.021372534159264083, 'n_estimators': 138, 'max_depth': 3, 'learning_rate': 0.0788176779144456, 'min_samples_leaf': 32, 'subsample': 0.9993202620699826, 'max_features': 0.7944324429031259}


In [17]:
gbm_study = optuna.load_study(
    study_name="gbm_study", 
    storage="sqlite:///optuna_studies/gbm_study.db" 
)
fig = vis.plot_param_importances(gbm_study)
fig.show()

In [36]:
#---Trial with the optimum model---
gbm_params = gbm_study.best_params.copy()
print(gbm_params)
loe_sigma_gbm = gbm_params.pop("looe_sigma", None)
gbm_best = GradientBoostingClassifier(
    **gbm_params, 
    random_state=42          
)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_gbm)
pipe_gbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    gbm_best
)
score_cv = cross_validate(pipe_gbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK gbm optimum: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the underfitted model---
gbm_best.set_params(n_estimators = 50)
pipe_gbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    gbm_best
)
score_cv = cross_validate(pipe_gbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK gbm underfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

#---Trial with the overfitted model---
gbm_best.set_params(n_estimators = 500)
pipe_gbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    gbm_best
)
score_cv = cross_validate(pipe_gbm, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
print(f"QWK gbm overfitted: train({np.mean(score_cv['train_QWK'])}), test({np.mean(score_cv['test_QWK'])})")

{'looe_sigma': 0.021372534159264083, 'n_estimators': 138, 'max_depth': 3, 'learning_rate': 0.0788176779144456, 'min_samples_leaf': 32, 'subsample': 0.9993202620699826, 'max_features': 0.7944324429031259}
QWK gbm optimum: train(0.54463247298519), test(0.4175210314352783)
QWK gbm underfitted: train(0.5223807364986979), test(0.3912179691683167)
QWK gbm overfitted: train(0.5597238831376765), test(0.39470585864163193)


## Voting and stacking classifiers

In [4]:
final_scoring = {
    "f1_macro": "f1_macro",
    "balanced_acc": "balanced_accuracy",
    "mcc": "matthews_corrcoef",
    "QWK": qwk
}

results = []

### Optuna hyperparameters loading

In [8]:
#We load the optuna studies for getting the optimized hyperparameters
rf_study = optuna.load_study(
    study_name="rf_study2", 
    storage="sqlite:///optuna_studies/rf_study.db" 
)

lgbm_study = optuna.load_study(
    study_name="lgbm_study1", 
    storage="sqlite:///optuna_studies/lgbm_study.db" 
)

cat_study = optuna.load_study(
    study_name="cat_study_native", 
    storage="sqlite:///optuna_studies/cat_study.db"
)

xgb_study = optuna.load_study(
    study_name="xgb_study2", 
    storage="sqlite:///optuna_studies/xgb_study.db")

gbm_study = optuna.load_study(
    study_name="gbm_study",
    storage="sqlite:///optuna_studies/gbm_study.db" 
)

### Models initialization

In [6]:
# Custom Wrapper to handle CatBoost's native categorical features 
# and avoid Scikit-Learn's cross-validation routing errors.
class CatBoostWrapper(ClassifierMixin, BaseEstimator):
    def __init__(
        self,
        cat_features=None,
        learning_rate=0.03,
        max_depth=6,
        n_estimators=1000,
        l2_leaf_reg=3.0,
        random_strength=1,
        bagging_temperature=1,
        border_count=128,
        allow_writing_files=False,
        random_state=42,
        thread_count=-1,
        verbose=0
    ):
        self.cat_features = cat_features
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.l2_leaf_reg = l2_leaf_reg
        self.random_strength = random_strength
        self.bagging_temperature = bagging_temperature
        self.border_count = border_count
        self.allow_writing_files = allow_writing_files
        self.random_state = random_state
        self.thread_count = thread_count
        self.verbose = verbose

    def fit(self, X, y, **fit_params):
        # Instantiate the actual model with the initialized parameters
        self.model_ = CatBoostClassifier(
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            n_estimators=self.n_estimators,
            l2_leaf_reg=self.l2_leaf_reg,
            random_strength=self.random_strength,
            bagging_temperature=self.bagging_temperature,
            border_count=self.border_count,
            allow_writing_files=self.allow_writing_files,
            random_state=self.random_state,
            thread_count=self.thread_count,
            verbose=self.verbose
        )
        # Pass cat_features explicitly during fit to bypass sklearn restriction
        self.model_.fit(X, y, cat_features=self.cat_features)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    @property
    def classes_(self):
        # Required by sklearn ensembles (like Stacking/Voting)
        return self.model_.classes_

cat_params = cat_study.best_params.copy()
cat_params.pop("looe_sigma", None)
# Remove cat_features from params to prevent duplication in the wrapper
cat_params.pop("cat_features", None)
cat_params.update({
    'random_strength': 1, 
    'bagging_temperature': 1, 
    'border_count': 128, 
    'allow_writing_files': False, 
    'random_state': 42, 
    'thread_count': -1,
    'verbose': 0,
})

# Instantiate using the Custom Wrapper
cat_best = CatBoostWrapper(
    cat_features=list(categorical_cols),
    **cat_params
)

#RandomForest
rf_params = rf_study.best_params.copy()
print(rf_params)
# Remove preprocessor-specific param to avoid argument errors
loe_sigma_rf = rf_params.pop("looe_sigma", None)

rf_best = RandomForestClassifier(
    **rf_params,    
    bootstrap=True,            
    class_weight="balanced",    
    random_state=42,
    n_jobs=-1
)

# XGBoost
xgb_params = xgb_study.best_params.copy()
print(xgb_params)
loe_sigma_xgb = xgb_params.pop("looe_sigma", None)
xgb_best = XGBClassifier(
    **xgb_params,
    random_state=42,         
    n_jobs=-1,
    eval_metric="mlogloss"  
)

# LightGBM
lgbm_params = lgbm_study.best_params.copy()
print(lgbm_params)
loe_sigma_lgbm = lgbm_params.pop("looe_sigma", None)

lgbm_best = LGBMClassifier(
    **lgbm_params,  
    subsample_freq=1,          
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Gradient Boosting
gbm_params = gbm_study.best_params.copy()
print(gbm_params)
loe_sigma_gbm = gbm_params.pop("looe_sigma", None)

gbm_best = GradientBoostingClassifier(
    **gbm_params, 
    random_state=42          
)

{'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
{'looe_sigma': 0.010773764065091765, 'n_estimators': 552, 'max_depth': 10, 'learning_rate': 0.01294031155889707, 'subsample': 0.6459278279430541, 'colsample_bytree': 0.6169935895547909, 'min_child_weight': 1}
{'looe_sigma': 0.017274353532403468, 'n_estimators': 431, 'max_depth': 14, 'learning_rate': 0.021117244921237944, 'num_leaves': 49, 'min_child_samples': 8, 'subsample': 0.7392171099865767, 'colsample_bytree': 0.9999202226086774}
{'looe_sigma': 0.021372534159264083, 'n_estimators': 138, 'max_depth': 3, 'learning_rate': 0.0788176779144456, 'min_samples_leaf': 32, 'subsample': 0.9993202620699826, 'max_features': 0.7944324429031259}


In [7]:
# Random Forest (RandomOverSampler)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_rf)
pipe_rf = make_pipeline(
    clone(preprocessorLOEEALL), 
    rf_best
)

# LightGBM (RandomOverSampler)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_lgbm)
pipe_lgbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42), 
    lgbm_best
)

# Gradient Boosting (auto balanced)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_gbm)
pipe_gbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    gbm_best
)

#XGBoost (RandomOverSampler)
preprocessorLOEEALL.set_params(LOOE__sigma=loe_sigma_xgb)
pipe_xgb = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    xgb_best
)

### Optimized models results

In [8]:
modelos_individuales = {
    "Random Forest": pipe_rf,
    "LightGBM": pipe_lgbm,
    "CatBoost": cat_best,
    "XGBoost": pipe_xgb,
    "Gradient Boosting": pipe_gbm
}

for nombre, modelo in modelos_individuales.items():
    print(f"Evaluando: {nombre}...")
    
    results_cv = cross_validate(
        modelo, 
        X_train, 
        y_train, 
        cv=skf, 
        scoring=final_scoring, 
        n_jobs=1,
        return_train_score = True
    )

    result = {
        "Model": f"{nombre} optimized",

        # F1 Macro
        "f1_mean_test": np.mean(results_cv["test_f1_macro"]),
        "f1_mean_train": np.mean(results_cv["train_f1_macro"]),
        "f1_sd_test": np.std(results_cv["test_f1_macro"]),
        "f1_sd_train": np.std(results_cv["train_f1_macro"]),

        # QWK (Quadratic Weighted Kappa)
        "qwk_mean_test": np.mean(results_cv["test_QWK"]),
        "qwk_mean_train": np.mean(results_cv["train_QWK"]),
        "qwk_sd_test": np.std(results_cv["test_QWK"]),
        "qwk_sd_train": np.std(results_cv["train_QWK"]),

        # Balanced Accuracy
        "balanced_acc_mean_test": np.mean(results_cv["test_balanced_acc"]),
        "balanced_acc_mean_train": np.mean(results_cv["train_balanced_acc"]),
        "balanced_acc_sd_test": np.std(results_cv["test_balanced_acc"]),
        "balanced_acc_sd_train": np.std(results_cv["train_balanced_acc"]),

        # MCC (Matthews Correlation Coefficient)
        "mcc_mean_test": np.mean(results_cv["test_mcc"]),
        "mcc_mean_train": np.mean(results_cv["train_mcc"]),
        "mcc_sd_test": np.std(results_cv["test_mcc"]),
        "mcc_sd_train": np.std(results_cv["train_mcc"])
    }
    results.append(result)
    
    # Calculamos la media del QWK
    qwk_medio = np.mean(results_cv['test_QWK'])
    
    print(f"QWK Promedio: {qwk_medio:.4f}\n")

Evaluando: Random Forest...
QWK Promedio: 0.4173

Evaluando: LightGBM...
QWK Promedio: 0.4234

Evaluando: CatBoost...
QWK Promedio: 0.4434

Evaluando: XGBoost...
QWK Promedio: 0.4201

Evaluando: Gradient Boosting...
QWK Promedio: 0.4175



### Voting classifier

In [ ]:
combinaciones_pesos = [
    [1, 2, 5, 2, 1],
    [0, 2, 6, 2, 0],
    [1, 3, 5, 3, 1]
]

for pesos in combinaciones_pesos:
    print(f"Pesos: {pesos}")
    voting_clf = VotingClassifier(
        estimators=[
            ('rf', pipe_rf),
            ('lgbm', pipe_lgbm),
            ('cat', cat_best),
            ('xgb', pipe_xgb),
            ('gbm', pipe_gbm)
        ],
        voting='soft',   
        weights=pesos
    )
    results_cv = cross_validate(voting_clf, X_train, y_train, cv=skf, scoring=scoring)

    qwk_medio = np.mean(results_cv['test_QWK'])

    print(f"QWK obtenido: {qwk_medio:.4f}")

Pesos: [1, 2, 5, 2, 1]
QWK obtenido: 0.4435
Pesos: [0, 2, 6, 2, 0]
QWK obtenido: 0.4429
Pesos: [1, 3, 5, 3, 1]
QWK obtenido: 0.4399


In [9]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', cat_best),
        ('xgb', pipe_xgb),
        ('gbm', pipe_gbm)
    ],
    voting='soft',   
    weights=[1, 2, 5, 2, 1]
)
results_cv = cross_validate(voting_clf, X_train, y_train, cv=skf, scoring=final_scoring, return_train_score=True)

result = {
    "Model": f"Voting classifier Soft (weights = [1, 2, 5, 2, 1])",

    # F1 Macro
    "f1_mean_test": np.mean(results_cv["test_f1_macro"]),
    "f1_mean_train": np.mean(results_cv["train_f1_macro"]),
    "f1_sd_test": np.std(results_cv["test_f1_macro"]),
    "f1_sd_train": np.std(results_cv["train_f1_macro"]),

    # QWK (Quadratic Weighted Kappa)
    "qwk_mean_test": np.mean(results_cv["test_QWK"]),
    "qwk_mean_train": np.mean(results_cv["train_QWK"]),
    "qwk_sd_test": np.std(results_cv["test_QWK"]),
    "qwk_sd_train": np.std(results_cv["train_QWK"]),

    # Balanced Accuracy
    "balanced_acc_mean_test": np.mean(results_cv["test_balanced_acc"]),
    "balanced_acc_mean_train": np.mean(results_cv["train_balanced_acc"]),
    "balanced_acc_sd_test": np.std(results_cv["test_balanced_acc"]),
    "balanced_acc_sd_train": np.std(results_cv["train_balanced_acc"]),

    # MCC (Matthews Correlation Coefficient)
    "mcc_mean_test": np.mean(results_cv["test_mcc"]),
    "mcc_mean_train": np.mean(results_cv["train_mcc"]),
    "mcc_sd_test": np.std(results_cv["test_mcc"]),
    "mcc_sd_train": np.std(results_cv["train_mcc"])
}
results.append(result)

### Stacking classifier

In [10]:
# Custom wrapper to use Ridge Regression for ordinal classification tasks.
# It predicts continuous values and maps them back to valid discrete classes.
class RegresorOrdinalRedondeado(ClassifierMixin, BaseEstimator):
    _estimator_type = "classifier"

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        
    def fit(self, X, y):
        self.classes_ = np.unique(y)

        # Initialize and train the underlying Ridge regression model
        self.model_ = Ridge(alpha=self.alpha)
        self.model_.fit(X, y)
        return self
        
    def predict(self, X):
        # Make continuous predictions (ej: 1.84)
        predicciones_continuas = self.model_.predict(X)
        
        # Round the continuous values to the nearest integer (e.g., 2.0)
        predicciones_redondeadas = np.round(predicciones_continuas)
        
        # Clip the rounded values to ensure they stay within the known class boundaries 
        # (e.g., preventing out-of-bounds predictions like -1 or 5)
        predicciones_finales = np.clip(
            predicciones_redondeadas, 
            self.classes_.min(), 
            self.classes_.max()
        )
        return predicciones_finales.astype(int)

In [ ]:
# Split the data into a training and validation set
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.3, 
    random_state=42, 
    stratify=y_train  # importante para mantener proporción de clases
)

# Dictionary of candidates for the final meta-learner
meta_classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Ridge Classifier (Nativo)": RidgeClassifier(random_state=42),
    "Linear SVC": LinearSVC(max_iter=2000, random_state=42, dual=False),
    "Random Forest (Shallow)": RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    "Ridge Regresion (Ordinal)": RegresorOrdinalRedondeado(alpha=1.0)
}

resultados_stacking = {}

# Iterate over each meta-classifier to find the best performing one
for nombre, meta_clf in meta_classifiers.items():
    print(f"Entrenando con: {nombre}...")
    inicio = time.time()
    
    # Build the Stacking model using the current meta-estimator
    stacking_clf = StackingClassifier(
        estimators=[
            ('rf', pipe_rf),
            ('lgbm', pipe_lgbm),
            ('cat', cat_best),
            ('xgb', pipe_xgb),
            ('gbm', pipe_gbm)
        ],
        final_estimator=meta_clf, 
        cv=skf, 
        passthrough=False,
        n_jobs=-1 
    )
    
    # Train the ensemble and make predictions on the validation set
    stacking_clf.fit(X_tr, y_tr)
    y_pred = stacking_clf.predict(X_val)
    
    qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')
    tiempo = time.time() - inicio
    
    resultados_stacking[nombre] = qwk
    print(f"QWK: {qwk:.4f} (Tiempo: {tiempo:.1f}s)\n")

Entrenando con: Logistic Regression...
QWK: 0.4448 (Tiempo: 527.9s)

Entrenando con: Ridge Classifier (Nativo)...
QWK: 0.4285 (Tiempo: 550.8s)

Entrenando con: Linear SVC...
QWK: 0.4269 (Tiempo: 534.3s)

Entrenando con: Random Forest (Shallow)...
QWK: 0.4327 (Tiempo: 522.1s)

Entrenando con: Ridge Regresion (Ordinal)...
QWK: 0.4067 (Tiempo: 500.9s)



#### Optimizacion stacking con logistic

In [14]:
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', cat_best),
        ('xgb', pipe_xgb),
        ('gbm', pipe_gbm)
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42), 
    cv=skf, 
    passthrough=False,
    n_jobs=1
)

param_grid = {
    'final_estimator__C': [0.01, 0.1, 1.0, 10.0]
}

grid_stacking = GridSearchCV(
    stacking_clf,
    param_grid,
    cv=3, 
    scoring=final_scoring, 
    refit="QWK",
    return_train_score=True,
    n_jobs=1,
    verbose=2
)

grid_stacking.fit(X_train, y_train)

best_idx = grid_stacking.best_index_
best_C = grid_stacking.best_params_['final_estimator__C']
results_cv = grid_stacking.cv_results_

for i, param in enumerate(results_cv['params']):
    c_val = param['final_estimator__C']
    qwk_test = results_cv['mean_test_QWK'][i]
    qwk_train = results_cv['mean_train_QWK'][i]
    print(f"Prueba {i+1} | C={c_val:<6} -> Test QWK: {qwk_test:.4f} (Train: {qwk_train:.4f})")

result = {
    "Model": f"Stacking Classifier with Logistic Regression (GridSearch C={best_C})",

    # F1 Macro (Asegúrate de que los nombres coinciden con tu diccionario final_scoring)
    "f1_mean_test": results_cv["mean_test_f1_macro"][best_idx],
    "f1_mean_train": results_cv["mean_train_f1_macro"][best_idx],
    "f1_sd_test": results_cv["std_test_f1_macro"][best_idx],
    "f1_sd_train": results_cv["std_train_f1_macro"][best_idx],

    # QWK
    "qwk_mean_test": results_cv["mean_test_QWK"][best_idx],
    "qwk_mean_train": results_cv["mean_train_QWK"][best_idx],
    "qwk_sd_test": results_cv["std_test_QWK"][best_idx],
    "qwk_sd_train": results_cv["std_train_QWK"][best_idx],

    # Balanced Accuracy
    "balanced_acc_mean_test": results_cv["mean_test_balanced_acc"][best_idx],
    "balanced_acc_mean_train": results_cv["mean_train_balanced_acc"][best_idx],
    "balanced_acc_sd_test": results_cv["std_test_balanced_acc"][best_idx],
    "balanced_acc_sd_train": results_cv["std_train_balanced_acc"][best_idx],

    # MCC
    "mcc_mean_test": results_cv["mean_test_mcc"][best_idx],
    "mcc_mean_train": results_cv["mean_train_mcc"][best_idx],
    "mcc_sd_test": results_cv["std_test_mcc"][best_idx],
    "mcc_sd_train": results_cv["std_train_mcc"][best_idx]
}
results.append(result)

print(f"Optimization complete. The best C was {best_C}")
print(f"Stacking QWK: {results_cv['mean_test_QWK'][best_idx]:.4f}")

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END ............................final_estimator__C=0.01; total time=12.3min
[CV] END ............................final_estimator__C=0.01; total time=11.7min
[CV] END ............................final_estimator__C=0.01; total time=12.0min
[CV] END .............................final_estimator__C=0.1; total time=11.7min
[CV] END .............................final_estimator__C=0.1; total time=11.7min
[CV] END .............................final_estimator__C=0.1; total time=12.5min
[CV] END .............................final_estimator__C=1.0; total time=12.7min
[CV] END .............................final_estimator__C=1.0; total time=12.7min
[CV] END .............................final_estimator__C=1.0; total time=12.7min
[CV] END ............................final_estimator__C=10.0; total time=13.8min
[CV] END ............................final_estimator__C=10.0; total time=14.4min
[CV] END ............................final_estima

In [15]:
print(results)

[{'Model': 'Random Forest optimized', 'f1_mean_test': np.float64(0.3346712176291856), 'f1_mean_train': np.float64(0.6223677318355245), 'f1_sd_test': np.float64(0.0027523975873993806), 'f1_sd_train': np.float64(0.011476442514378408), 'qwk_mean_test': np.float64(0.41728014610265063), 'qwk_mean_train': np.float64(0.6586526363597247), 'qwk_sd_test': np.float64(0.007279972884147533), 'qwk_sd_train': np.float64(0.0028323702911812808), 'balanced_acc_mean_test': np.float64(0.3486978375012761), 'balanced_acc_mean_train': np.float64(0.5973256566469718), 'balanced_acc_sd_test': np.float64(0.0020359099587588), 'balanced_acc_sd_train': np.float64(0.009063427648989089), 'mcc_mean_test': np.float64(0.25452369879731496), 'mcc_mean_train': np.float64(0.5746867886485801), 'mcc_sd_test': np.float64(0.004620937847425031), 'mcc_sd_train': np.float64(0.007271611612027791)}, {'Model': 'LightGBM optimized', 'f1_mean_test': np.float64(0.3431285723577543), 'f1_mean_train': np.float64(0.597503825822072), 'f1_sd_

### Copy all results on a csv

In [16]:
final_results = pd.DataFrame(results)
final_results.to_csv("optimized_results.csv")